# Week 8 — Visualisation & Putting It Together: Building a Streamlit Dashboard
## Phase 2a Python | PORA Academy Cohort 7

Yesterday you turned Olist data into charts with matplotlib and seaborn. Today is the
grand finale of Phase 2a: we wrap those charts, KPIs, and filters into a **real web
application** that a manager could open in a browser — no notebook required.

By the end of this session, you will be able to:
- Understand what Streamlit is and why it matters for data analysts
- Build a working multi-page dashboard using Streamlit
- Use core Streamlit components: `st.title`, `st.metric`, `st.selectbox`, `st.columns`, `st.pyplot`, `st.dataframe`, `@st.cache_data`
- Run a Streamlit app from Google Colab using pyngrok
- Know the Streamlit Community Cloud deployment workflow (used in Phase 2c Capstone)

## The Business Question

In the Olist dataset, the operations team does not want to open Colab and scroll through
code every Monday morning. They want a **live dashboard**: pick a year, and instantly see
that 2017 had **45,101 orders** at a **96.3% delivered rate**, while 2018 grew to
**54,011 orders** at **97.7% delivered** — with charts for monthly volume, top states, and
payment methods updating on the spot.

A Jupyter notebook cannot do that for a non-technical stakeholder. To turn our analysis into
something a manager can click through in a browser, we need **Streamlit** — a Python library
that converts an ordinary `.py` script into an interactive web app with zero HTML, CSS, or
JavaScript. Everything you already know about pandas and matplotlib carries straight over.

## Setup — Load the Olist Data

Run this cell first. It mounts Google Drive and loads the Olist tables. Every later cell —
and the sanity-check that verifies our dashboard's numbers — depends on the variables created
here.

In [ ]:
import pandas as pd
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
olist_path = '/content/drive/MyDrive/olist-data'

# Load all datasets
orders = pd.read_csv(os.path.join(olist_path, 'olist_orders_dataset.csv'))
customers = pd.read_csv(os.path.join(olist_path, 'olist_customers_dataset.csv'))
order_items = pd.read_csv(os.path.join(olist_path, 'olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(olist_path, 'olist_products_dataset.csv'))
reviews = pd.read_csv(os.path.join(olist_path, 'olist_order_reviews_dataset.csv'))
payments = pd.read_csv(os.path.join(olist_path, 'olist_order_payments_dataset.csv'))
sellers = pd.read_csv(os.path.join(olist_path, 'olist_sellers_dataset.csv'))
product_translations = pd.read_csv(os.path.join(olist_path, 'product_category_name_translation.csv'))

# Verify datasets loaded
print(f"Loaded {len(orders):,} orders")        # expected: Loaded 99,441 orders
print(f"Loaded {len(customers):,} customers")   # expected: Loaded 99,441 customers
print(f"Loaded {len(payments):,} payments")     # expected: Loaded 103,886 payments

## Concept 1 — What is Streamlit?

**Streamlit turns a Python script into an interactive web app** — you write plain Python
top to bottom, and Streamlit renders each `st.` command as a widget or chart in the browser.
Think of it like a recipe card: you list the steps once, and every time someone opens the
kitchen (the app), Streamlit re-cooks the whole recipe from the top and plates the result.

That last part is the single most important idea to hold on to: **every time a user touches
a widget, Streamlit re-runs the entire script from line 1 to the bottom.** Change the year in
a dropdown, and the whole file executes again with the new value. This is why we cache slow
work (like loading CSVs) — otherwise the data would reload on every click.

You install Streamlit and the tunnelling helper once per Colab session with:

```python
!pip install streamlit pyngrok -q
```

A Streamlit app is never run with `python app.py`. Instead you write your code to a file and
launch it with `streamlit run app.py`. Below we use the Jupyter `%%writefile` magic to save a
tiny first app to disk — the magic just writes the cell's text to `app.py`, it does **not**
run Streamlit inside this notebook.

In [ ]:
%%writefile app.py
import streamlit as st

st.title("Hello from Streamlit!")
st.write("This is my first app.")
st.metric("Total Orders (2017)", "45,101")


## Concept 2 — Running Streamlit in Colab via pyngrok

Colab runs on a Google server with no visible browser, so it **cannot render a Streamlit app
directly**. The workaround is a three-part relay: (1) write the app to `app.py`, (2) start the
Streamlit server in the background on port 8501 with `subprocess`, and (3) open a public
tunnel to that port with **pyngrok**, which hands you a shareable `https://...ngrok.io` URL.

Think of pyngrok as a temporary doorway punched from the public internet straight to the
private server running inside your Colab session — anyone with the link can walk through and
see your live app.

The launch sequence below is meant to run **in Colab**, not inside this validation notebook
(it would try to install ngrok and open a network tunnel). Study it, then run it in your own
Colab session after grabbing a free auth token from [ngrok.com](https://ngrok.com):

```python
import subprocess
import time
from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")   # free token from ngrok.com

process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(3)                                    # give the server a moment to boot

public_url = ngrok.connect(8501)
print(f"App is live at: {public_url}")
```

The cell below simply confirms the pattern without touching the network, so this notebook stays
runnable end to end.

In [ ]:
# The four launch steps, kept as comments so this cell never touches the network:
#   1. import subprocess, time  and  from pyngrok import ngrok
#   2. ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN")
#   3. subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
#   4. public_url = ngrok.connect(8501)  ->  a public https://...ngrok.io link

launch_steps = ["write app.py", "start streamlit server", "open pyngrok tunnel", "share the URL"]
print(f"pyngrok relay has {len(launch_steps)} steps")
# expected: pyngrok relay has 4 steps

## Concept 3 — Core Components — Full Dashboard

Now we assemble the real thing. A production dashboard stacks a handful of core components:

- **`@st.cache_data`** — decorates the loader so the CSVs load **once**, not on every re-run.
- **`st.sidebar.selectbox`** — a year filter parked in the left sidebar.
- **`st.columns` + `st.metric`** — a row of four headline KPI cards.
- **`st.pyplot(fig)`** — embeds a matplotlib figure (exactly the charts you built yesterday).
- **two-column layout** — states chart on the left, payment breakdown on the right.
- **`st.checkbox` + `st.dataframe`** — a toggle that reveals a raw-data table on demand.

Read the app below like a script that runs top to bottom every time the user changes the year.
Because it lives inside `%%writefile app.py`, this cell only **writes** the file — it does not
import Streamlit or execute the dashboard here. We verify the numbers it will show in the next
section with plain pandas.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import os

DATA_DIR = "/content/drive/MyDrive/cohort7/datasets/olist"

st.set_page_config(page_title="Olist Dashboard", layout="wide")

st.title("Olist E-Commerce Dashboard")
st.markdown("**Brazilian marketplace orders — 2016 to 2018**")
st.divider()

@st.cache_data
def load_data():
    orders = pd.read_csv(os.path.join(DATA_DIR, "olist_orders_dataset.csv"))
    customers = pd.read_csv(os.path.join(DATA_DIR, "olist_customers_dataset.csv"))
    payments = pd.read_csv(os.path.join(DATA_DIR, "olist_order_payments_dataset.csv"))
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
    orders["year"] = orders["order_purchase_timestamp"].dt.year
    orders["month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
    return orders, customers, payments

orders, customers, payments = load_data()

st.sidebar.header("Filters")
year_options = sorted(orders["year"].dropna().unique().astype(int).tolist())
selected_year = st.sidebar.selectbox("Select Year", year_options, index=1)

filtered = orders[orders["year"] == selected_year]

col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Orders", f"{len(filtered):,}")
col2.metric("Delivered", f"{(filtered['order_status'] == 'delivered').sum():,}")
col3.metric("Canceled", f"{(filtered['order_status'] == 'canceled').sum():,}")
col4.metric("Delivered %", f"{(filtered['order_status'] == 'delivered').mean()*100:.1f}%")

st.subheader(f"Monthly Orders — {selected_year}")
monthly = filtered.groupby("month")["order_id"].count().reset_index()
monthly.columns = ["month", "orders"]

fig1, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(monthly["month"], monthly["orders"], marker="o", color="steelblue", linewidth=2)
ax1.set_xlabel("Month")
ax1.set_ylabel("Orders")
ax1.tick_params(axis="x", rotation=45)
plt.tight_layout()
st.pyplot(fig1)
plt.close()

col_left, col_right = st.columns(2)

with col_left:
    st.subheader("Top 10 States by Orders")
    merged = filtered.merge(customers[["customer_id", "customer_state"]], on="customer_id")
    state_counts = merged["customer_state"].value_counts().head(10)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.barh(state_counts.index[::-1], state_counts.values[::-1], color="coral")
    ax2.set_xlabel("Orders")
    plt.tight_layout()
    st.pyplot(fig2)
    plt.close()

with col_right:
    st.subheader("Payment Method Breakdown")
    pay_counts = payments["payment_type"].value_counts()
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    ax3.pie(pay_counts.values, labels=pay_counts.index, autopct="%1.1f%%",
            colors=["#3498db","#e74c3c","#2ecc71","#f39c12","#9b59b6"])
    plt.tight_layout()
    st.pyplot(fig3)
    plt.close()

if st.checkbox("Show raw data sample"):
    st.dataframe(filtered.head(20))


## Going Deeper — Trust, but Verify the KPI Numbers

Because the `app.py` above only gets written to disk (never executed here), how do we know its
KPI cards will show the *right* numbers? We reproduce the exact same calculation in plain pandas
on the `orders` DataFrame we loaded in setup. If our pandas result matches the verified figures,
the dashboard's `st.metric` cards will too — they run identical logic.

This "verify the number outside the app, then trust it inside the app" habit is exactly how you
debug a Streamlit dashboard: never eyeball a metric card, always reproduce it in a cell first.

In [ ]:
# Reproduce the four dashboard KPIs per year, directly on the orders DataFrame.
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['year'] = orders['order_purchase_timestamp'].dt.year

for yr in [2017, 2018]:
    yearly = orders[orders['year'] == yr]
    total = len(yearly)
    delivered = int((yearly['order_status'] == 'delivered').sum())
    canceled = int((yearly['order_status'] == 'canceled').sum())
    delivered_pct = (yearly['order_status'] == 'delivered').mean() * 100
    print(f"{yr}: Total={total:,} | Delivered={delivered:,} | Canceled={canceled:,} | Delivered%={delivered_pct:.1f}%")

# expected: 2017: Total=45,101 | Delivered=43,428 | Canceled=265 | Delivered%=96.3%
# expected: 2018: Total=54,011 | Delivered=52,783 | Canceled=334 | Delivered%=97.7%

## Common Mistakes

Three traps catch nearly every first-time Streamlit author. We show the wrong version as a
comment (so nothing crashes) and then the fix.

In [ ]:
# -- MISTAKE 1: forgetting @st.cache_data --------------------
# WRONG - reloads 100k+ rows from disk on EVERY widget click, app feels frozen:
# def load_data():
#     return pd.read_csv("olist_orders_dataset.csv")
#
# CORRECT - cache it so the CSV loads once and is reused on every re-run:
# @st.cache_data
# def load_data():
#     return pd.read_csv("olist_orders_dataset.csv")

# -- MISTAKE 2: running Streamlit like a normal script -------
# WRONG - this just prints warnings and shows nothing:
# !python app.py
# CORRECT - always launch with the streamlit command:
# !streamlit run app.py

# -- MISTAKE 3: passing st.pyplot() no figure ----------------
# WRONG - st.pyplot() with no argument grabs an empty global figure:
# st.pyplot()
# CORRECT - build an explicit fig, then hand it over:
# fig, ax = plt.subplots(); ax.bar(x, y); st.pyplot(fig)

print("Cache the loader, launch with 'streamlit run', and always pass st.pyplot a figure.")
# expected: Cache the loader, launch with 'streamlit run', and always pass st.pyplot a figure.

## Mini-Challenge

⏱ ~5 minutes

Two of the KPI cards in this week's **assignment** are *Avg Payment Value* and *Most Common
Payment Type*. Before wiring them into `st.metric`, compute them in plain pandas from the
`payments` table so you know what the cards should read.

**Expected output:**
```
Average payment value: R$154.10
Most common payment type: credit_card
```

Fill in the two blanks below, then uncomment the two print lines to check your work.

In [ ]:
# payments is already loaded in the setup cell - do not reload it.

# 1) Average of the payment_value column, rounded to 2 decimals
avg_payment = None  # Your code here

# 2) The single most common value in the payment_type column
top_payment_type = None  # Your code here

# print(f"Average payment value: R${avg_payment}")
# print(f"Most common payment type: {top_payment_type}")
# expected: Average payment value: R$154.10
# expected: Most common payment type: credit_card

## Part 3 — Group Exercise (40 min)

Each group extends the app above by adding one new section. Assign sections:

- **Group A**: Add a delivery time histogram. Load `order_delivered_customer_date`, compute `delivery_days`, plot a histogram. Expected: mean = 12.1 days, median = 10.0 days.
- **Group B**: Add a review score bar chart. Load `olist_order_reviews_dataset.csv`, plot `review_score` counts. Expected: 5-star = 57,328, 1-star = 11,424.
- **Group C**: Add a top-10 revenue categories chart. Requires merging `order_items` + `products` + translation. Expected: `health_beauty` = R$1,258,681.34.
- **Group D (challenge)**: Add a `st.selectbox` for state selection. Filter all charts by the selected state. Verify SP total orders (2017) = 17,760.

**Workflow for every group:** first *verify your target number in pandas* using the cell below,
then add the matching chart to `app.py` inside a `%%writefile` cell and re-launch. Verify first,
build second.

In [ ]:
# Verify YOUR group's target number here BEFORE adding the chart to app.py.
# orders, order_items, products, reviews, customers, product_translations, payments are all loaded.

# Group A - delivery time in days (mean and median):
# orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
# delivery_days = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
# Your code here -> expected mean = 12.1, median = 10.0

# Group B - review_score counts:
# Your code here -> expected 5-star = 57,328, 1-star = 11,424

# Group C - top revenue category (merge order_items + products + product_translations):
# Your code here -> expected health_beauty = R$1,258,681.34

# Group D - SP orders in 2017 (merge orders + customers, filter year and state):
# Your code here -> expected SP total orders (2017) = 17,760

print("Verify your group's number above, then add the chart to app.py and re-launch.")
# expected: Verify your group's number above, then add the chart to app.py and re-launch.

## From Colab to a Permanent Link — Streamlit Community Cloud

The pyngrok tunnel is perfect for a quick classroom demo, but the URL dies the moment your
Colab session ends. For a link you can put on a CV or send to a client, you deploy to
**Streamlit Community Cloud** — a free hosting service run by Streamlit. This is the workflow
you will follow in the Phase 2c Capstone:

1. Put your files in a public **GitHub** repo: `app.py`, a `requirements.txt` listing your
   libraries (`streamlit`, `pandas`, `matplotlib`), and your data (a small CSV, or code that
   downloads it).
2. Sign in at [share.streamlit.io](https://share.streamlit.io) with your GitHub account.
3. Click **New app**, pick the repo, branch, and `app.py`, then **Deploy**.
4. Streamlit installs your `requirements.txt`, runs the app, and gives you a permanent public
   `your-app.streamlit.app` URL that redeploys automatically every time you push to GitHub.

No servers, no `pyngrok`, no auth tokens — just push and share.

## Session Summary

| Concept | Key Syntax | Example |
|---|---|---|
| Write app to file | `%%writefile app.py` | first line of the cell |
| Cache slow loads | `@st.cache_data` | decorate `load_data()` |
| Headline KPIs | `st.columns` + `st.metric` | `col1.metric("Total Orders", "45,101")` |
| Sidebar filter | `st.sidebar.selectbox` | `st.sidebar.selectbox("Year", [2017, 2018])` |
| Embed a chart | `st.pyplot(fig)` | pass an explicit matplotlib figure |
| Toggle raw data | `st.checkbox` + `st.dataframe` | `if st.checkbox("Show data"): st.dataframe(df)` |
| Run in Colab | `subprocess.Popen` + `pyngrok` | tunnel port 8501 to a public URL |

**Deploying for real:** in Colab we tunnel with pyngrok, but for a permanent link you push
`app.py` + a `requirements.txt` to GitHub and connect the repo to **Streamlit Community Cloud**
(free) — one click gives you a public `streamlit.app` URL. You will do exactly this in the
Phase 2c Capstone.

---
**That's a wrap on Phase 2a Python!** You have gone from `print("hello")` in Week 1 to a live,
filterable web dashboard in Week 8. **Coming up — Phase 2c Capstone**: your group builds and
deploys a full Olist Streamlit app to Streamlit Community Cloud as your portfolio piece.